# 99 Veeva Link feature dictionary (Phase 4, O4)

Task O4 of `95_veeva_link_phase4_working_plan.md`. Builds one dictionary (P4-D8) from the code artefacts, not by hand (P4-R7): `features_phase1/feature_manifest` (T1 / T2 rules and status), `features_phase1/schema_lock/columns.json`, `features_phase2/schema_lock/t3_columns.json` and `cross_columns.json`. T3 and cross rules are written below from `82_`; their status is "wait: G2 open" until `t3_decisions` exists, with the Phase 2 / 3 reading added.

Checks: N10 every locked column has exactly one row; N11 every row has a source table and a rule; N12 status filled. Output: `OUT/docs/feature_dictionary/run=<ts>/` (parquet + csv).

In [0]:
import json, pandas as pd, datetime as dt
from pyspark.sql.types import StructType, StructField, StringType

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
lock = lambda p: json.loads("".join(r.value for r in spark.read.text(OUT + p).collect()))
LOCKS = {"T1/T2": lock("features_phase1/schema_lock/columns.json"), "T3": lock("features_phase2/schema_lock/t3_columns.json"),
         "cross": lock("features_phase2/schema_lock/cross_columns.json")}
man = spark.read.parquet(OUT + "features_phase1/feature_manifest").toPandas()

SRC_BY_FAMILY = {"payments": "PAYMENT", "trials": "CLINICAL_TRIAL, SCIENTIFIC_EXPERT_CLINICAL_TRIAL", "publications": "PUBLICATION, SCIENTIFIC_EXPERT_PUBLICATION",
                 "events": "MEDICAL_EVENT, EVENT_ATTENDEE", "profile": "SCIENTIFIC_EXPERT", "affiliation": "ASSOCIATION, ADDRESS",
                 "sponsorship": "COMPANY_SPONSORSHIP", "coverage": "SCIENTIFIC_EXPERT, CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA",
                 "scores": "CLINICAL_LEADER, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA",
                 "composite": "other T1 features (PAYMENT, PUBLICATION, MEDICAL_EVENT, ASSOCIATION)"}
KEYS = {"NPI_NUMBER": ("key", "HCP Universe / spine (54_)", "HCP NPI; one row per NPI per cohort"),
        "PROVIDER_ID": ("key", "HCP Universe / spine (54_)", "NBA provider id of the NPI"),
        "COHORT_ID": ("key", "cohort calendar (71_)", "Cohort, YYYY-MM_offset7; cut-off = month end + 6 days"),
        "t3_version": ("key", "82_", "snapshot = 2025-03-06 load for every cohort; asof = latest load on or before the cohort cut-off (null before 2024-08-01)"),
        "cohort_set": ("key", "cohort calendar (71_)", "Cohort set of the row (NBA 3.0 or NBA 3.1 cohorts)"),
        "veeva_window_end": ("key", "cohort_windows (71_)", "End date of the activity windows used for this cohort (see features_phase1/cohort_windows)"),
        "veeva_data_age_months": ("key", "cohort_windows (71_)", "Months between the Veeva data cut-off and the cohort cut-off"),
        "veeva_snapshot_id": ("key", "52_ snapshot", "Snapshot folder the features were built from (load=<hub load date>)"),
        "veeva_coverage_status": ("coverage", "spine + SCIENTIFIC_EXPERT + score tables (54_)", "SCORED / IN_VEEVA_UNSCORED / NOT_IN_VEEVA; tells a true 0 from no Veeva data")}
CL = "CLINICAL_LEADER"
T3 = {"veeva_best_decile": ("scores", f"{CL}, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA", "Best in-scope decile 1-10 across CL, FA, DR (focus_dictionary_v1); expert = 8-10"),
      "veeva_decile_bin": ("scores", f"{CL}, FOCUS_AREA, DIGITAL_REACH_FOCUS_AREA", "veeva_best_decile banded 1-3 / 4-7 / 8-10"),
      **{f"veeva_cl_decile_{k}": ("scores", CL, f"Max Clinical Leader decile for in-scope {v} focus names") for k, v in
         {"ckd": "CKD", "hf": "heart failure", "t2d": "type 2 diabetes", "t1d": "type 1 diabetes", "htn": "hypertension", "mra_class": "MRA drug class", "sglt2_class": "SGLT2 drug class"}.items()},
      "veeva_fa_decile_max_inscope": ("scores", "FOCUS_AREA", "Max Focus Area decile over in-scope focus names"),
      "veeva_dr_any_inscope": ("scores", "DIGITAL_REACH_FOCUS_AREA", "1 if any in-scope Digital Reach decile"),
      **{f"veeva_index_{k}": ("index", "INDEX", f"Veeva {k} index VALUE (0-10)") for k in ("publication", "trial", "event", "guideline")},
      "veeva_scientific_reach": ("index", "INDEX", "Scientific reach 0-4 (n/a, local, regional, national, global)"),
      "veeva_score_load_date": ("scores", "score load calendar (81_)", "Load date of the scores used"),
      "veeva_score_age_months": ("scores", "score load calendar (81_)", "Months between score load date and cohort cut-off")}
CROSS = {"veeva_idn_expert_share": ("cross", "T3 + mdm_stg_acct_hrchy_ker (82_)", "Share of other HCPs in the same IDN who are Veeva experts (decile 8-10)"),
         "veeva_idn_expert_share_zero_flag": ("cross", "T3 + mdm_stg_acct_hrchy_ker (82_)", "1 if the HCP has an IDN and no expert peers"),
         "veeva_practice_expert_peer_flag": ("cross", "T3 + cust_mdm_best_affl_publish_pd_outp_static (82_)", "1 if another HCP at the same practice (BHO_ID) is an expert"),
         "veeva_practice_n_peers": ("cross", "cust_mdm_best_affl_publish_pd_outp_static (82_)", "Number of other HCPs at the same practice"),
         "veeva_idn_n_peers": ("cross", "mdm_stg_acct_hrchy_ker (82_)", "Number of other HCPs in the same IDN")}
READING = {"veeva_best_decile": "R4 monotonic, 2.97x; R2 E5 candidate (2.22x); used in B1, B4",
           "veeva_cl_decile_ckd": "R2 E5 candidate (1.86x); B2 DISEASE_CKD", "veeva_cl_decile_t2d": "R2 E5 candidate (1.77x); B2 DISEASE_T2D",
           "veeva_cl_decile_htn": "R2 E5 candidate (1.35x)", "veeva_fa_decile_max_inscope": "R2 E5 candidate (1.63x)",
           "veeva_cl_decile_hf": "R2 no signal; B2.1 below segment average: drop candidate",
           "veeva_cl_decile_mra_class": "R2 no signal within claims bands; messaging hint only (B2)", "veeva_cl_decile_sglt2_class": "R2 no signal (1.03x); messaging hint only (B2)",
           "veeva_cl_decile_t1d": "R2 no signal", "veeva_scientific_reach": "R2 no signal",
           "veeva_idn_expert_share": "KER-10 test: keep as E7 candidate", "veeva_idn_expert_share_zero_flag": "KER-10 test: keep", "veeva_practice_expert_peer_flag": "KER-10 test: keep"}

rows = []
for tier, cols in LOCKS.items():
    for c in cols:
        rows.append({"column": c, "lock": tier})
d = pd.DataFrame(rows).groupby("column", sort=False).lock.apply(lambda s: ", ".join(s)).reset_index()
mi = man.set_index("feature")
out = []
for r in d.itertuples():
    c = r.column
    if c in KEYS:
        fam, src, rule = KEYS[c]; tier, status, reading = "key", "in use", ""
    elif c in mi.index:
        m = mi.loc[c]; fam, rule, tier = m.family, m.rule, m.tier
        src, status, reading = SRC_BY_FAMILY.get(m.family), m.status, f"kept={m.kept}; E1-E4 use: {''.join(str(int(bool(m[e]))) for e in ['E1', 'E2', 'E3', 'E4'])}"
    elif c in T3 or c in CROSS:
        fam, src, rule = (T3 | CROSS)[c]; tier = "T3" if c in T3 else "cross"
        status, reading = "wait: G2 open (business rules only, P2-D3)", READING.get(c, "")
    else:
        fam = src = rule = status = reading = None; tier = None
    out.append({"column": c, "locks": r.lock, "tier": tier, "family": fam, "source_tables": src, "rule": rule, "status": status, "phase2_3_reading": reading,
                "null_meaning": "null = no Veeva data (see veeva_coverage_status); 0 = in Veeva, no activity" if tier in ("T1", "T2", "T3", "cross") else "",
                "data_cutoff": "2025-03-06 (hub 2025-06-30)", "run_ts": RUN_TS})
dic = pd.DataFrame(out)
all_locked = [c for cols in LOCKS.values() for c in cols]
chk = pd.DataFrame([
    {"check": "N10", "name": "every locked column has exactly one row", "value": f"{len(set(all_locked))} distinct locked / {len(dic)} rows / dup {int(dic.column.duplicated().sum())}",
     "pass": set(all_locked) == set(dic.column) and not dic.column.duplicated().any()},
    {"check": "N11", "name": "every row has source tables and rule", "value": str(dic.loc[dic.source_tables.isna() | dic.rule.isna(), "column"].tolist()),
     "pass": bool(dic.source_tables.notna().all() and dic.rule.notna().all())},
    {"check": "N12", "name": "status filled", "value": str(dic.loc[dic.status.isna(), "column"].tolist()), "pass": bool(dic.status.notna().all())}])
chk["run_ts"] = RUN_TS
sp = lambda x: spark.createDataFrame(x.astype(object).where(x.notna(), None).map(lambda v: None if v is None else str(v)),
                                     StructType([StructField(c, StringType(), True) for c in x.columns]))
P = f"{OUT}docs/feature_dictionary/run={RUN_TS}/"
sp(dic).write.mode("overwrite").parquet(P + "dictionary")
sp(dic).coalesce(1).write.mode("overwrite").option("header", True).csv(P + "dictionary_csv")
sp(chk).write.mode("overwrite").parquet(P + "checks")
print(f"rows {len(dic)} | by tier: {dic.tier.value_counts(dropna=False).to_dict()} | written: {P}")
display(chk)
display(dic.groupby(["tier", "status"], dropna=False).size().reset_index(name="columns"))